<a href="https://colab.research.google.com/github/Neeti1987/Pathway-GEM-scripts/blob/main/Arginine_pathway_specific_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === STEP 1: Install dependencies ===
!pip install cobra pandas

import cobra
from cobra import Model, Reaction, Metabolite
import pandas as pd
from google.colab import files

# === STEP 2: Upload KO annotation file ===
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# === STEP 3: Parse KO annotations genome-wise ===
genome_kos = {}
with open(filename, "r") as f:
    for line in f:
        if not line.strip(): continue
        parts = line.strip().split("\t")
        if len(parts) < 2: continue
        genome_id, ko = parts[0], parts[1]
        genome_prefix = genome_id.split("_")[0]
        genome_kos.setdefault(genome_prefix, set()).add(ko)

# === STEP 4: KO logic for M00844 (Arginine biosynthesis, ornithine => arginine) ===
def check_arginine_module(ko_list):
    status = {}
    status["Ornithine_carbamoyltransferase"] = ("K00611" in ko_list)
    status["Argininosuccinate_synthase"] = ("K01940" in ko_list)
    status["Argininosuccinate_lyase"] = any(k in ko_list for k in ["K01755","K14681"])
    complete = all(status.values())
    return complete, status

# === STEP 5: Build arginine pathway model (permissive exchanges) ===
def build_arginine_model():
    model = Model("Arginine_pathway")

    # Metabolites (cytosolic, KEGG C numbers)
    carb_phos = Metabolite("C00169_c", compartment="c")   # Carbamoyl phosphate
    orn = Metabolite("C00077_c", compartment="c")        # L-Ornithine
    pi = Metabolite("C00009_c", compartment="c")         # Orthophosphate
    citrulline = Metabolite("C00327_c", compartment="c") # L-Citrulline
    atp = Metabolite("C00002_c", compartment="c")        # ATP
    asp = Metabolite("C00049_c", compartment="c")        # L-Aspartate
    amp = Metabolite("C00020_c", compartment="c")        # AMP
    ppi = Metabolite("C00013_c", compartment="c")        # Diphosphate
    argsucc = Metabolite("C03406_c", compartment="c")    # Argininosuccinate
    arg = Metabolite("C00062_c", compartment="c")        # L-Arginine
    fumarate = Metabolite("C00122_c", compartment="c")   # Fumarate

    # Reactions
    r01398 = Reaction("R01398")
    r01398.add_metabolites({carb_phos:-1, orn:-1, pi:1, citrulline:1})

    r01954 = Reaction("R01954")
    r01954.add_metabolites({atp:-1, citrulline:-1, asp:-1,
                            amp:1, ppi:1, argsucc:1})

    r01086 = Reaction("R01086")
    r01086.add_metabolites({argsucc:-1, arg:1, fumarate:1})

    model.add_reactions([r01398, r01954, r01086])

    # Exchanges for all metabolites (permissive style)
    for met in [carb_phos, orn, pi, citrulline, atp, asp, amp, ppi, argsucc, arg, fumarate]:
        ex = Reaction(f"EX_{met.id}")
        ex.add_metabolites({met:1})
        ex.lower_bound = -1000
        ex.upper_bound = 1000
        model.add_reactions([ex])

    # Demand reaction for arginine
    dm_arg = Reaction("DM_C00062")
    dm_arg.add_metabolites({arg:-1})
    dm_arg.lower_bound = 0
    dm_arg.upper_bound = 1000
    model.add_reactions([dm_arg])
    model.objective = dm_arg

    return model

# === STEP 6: Run genome-wise analysis ===
summary = []
for genome, kos in genome_kos.items():
    complete, status = check_arginine_module(kos)
    flux_value = None
    if complete:
        model = build_arginine_model()
        sol = model.optimize()
        flux_value = sol.objective_value
    summary.append({
        "Genome": genome,
        **status,
        "Module_complete": complete,
        "Flux_to_Arginine": flux_value
    })

# === STEP 7: Output results ===
df = pd.DataFrame(summary)
print(df)
df.to_csv("arginine_flux_genomewise.tsv", sep="\t", index=False)

# === STEP 8: Download TSV ===
files.download("arginine_flux_genomewise.tsv")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.8/141.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 64.9 MB/s eta 0:00:00


Saving FINAL BLAST.tsv to FINAL BLAST.tsv
    Genome  Ornithine_carbamoyltransferase  Argininosuccinate_synthase  \
0    ADCAI                            True                        True   
1    AFCAI                            True                        True   
2      BTB                            True                        True   
3      BTQ                            True                        True   
4   BTQVLC                            True                        True   
5     BTZ1                            True                        True   
6    China                            True                        True   
7   China1                            True                        True   
8    MEAM1                            True                        True   
9     PeMo                            True                        True   
10    SiSi                            True                        True   
11      TV                            True                        True

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>